# Оценка качества RAG-поиска

Анализ качества векторного поиска по записям дневника.

**Методология:**
- Синтетический тест-сет: первая половина записи → запрос, ищем оригинал
- Метрики: Precision@1, Precision@3, Precision@5, MRR
- Визуализация распределения косинусных расстояний
- Анализ ошибок (false positives / missed retrievals)

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('../.env')
print('OK')

In [ ]:
import asyncio
import asyncpg

async def load_entries(user_id: int):
    conn = await asyncpg.connect(os.getenv('DATABASE_URL').replace('+asyncpg', ''))
    rows = await conn.fetch(
        "SELECT id, text, created_at, embedding::text FROM entries "
        "WHERE user_id=$1 AND embedding IS NOT NULL AND text IS NOT NULL "
        "AND length(text) > 60 ORDER BY created_at",
        user_id
    )
    await conn.close()
    return rows

USER_ID = int(input('user_id: '))
rows = asyncio.run(load_entries(USER_ID))
print(f'Загружено {len(rows)} записей')

In [ ]:
import ast

entry_ids   = [r['id'] for r in rows]
texts       = [r['text'] for r in rows]
embeddings  = np.array([ast.literal_eval(r['embedding']) for r in rows], dtype=np.float32)

# Нормализуем для косинусного расстояния
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / np.clip(norms, 1e-9, None)

print(f'Матрица эмбеддингов: {embeddings.shape}')

## 1. Синтетический тест-сет

In [ ]:
from ai.embeddings import get_embedding

# Запрос = первые 40% текста записи, ответ = эмбеддинг полной записи
test_queries   = []
test_targets   = []   # индекс правильной записи

for i, text in enumerate(texts):
    words = text.split()
    if len(words) < 10:
        continue
    query_text = ' '.join(words[:max(5, len(words) * 2 // 5)])
    query_emb  = get_embedding(query_text)
    test_queries.append(np.array(query_emb, dtype=np.float32))
    test_targets.append(i)

print(f'Тест-сет: {len(test_queries)} запросов')

## 2. Поиск и вычисление метрик

In [ ]:
def cosine_search(query_emb: np.ndarray, top_k: int = 5) -> list[int]:
    """Возвращает индексы top-k записей по косинусному сходству"""
    q = query_emb / np.linalg.norm(query_emb)
    scores = embeddings_norm @ q
    return np.argsort(scores)[::-1][:top_k].tolist()

ks = [1, 3, 5]
precision = {k: 0 for k in ks}
reciprocal_ranks = []
top5_distances   = []

for query_emb, target_idx in zip(test_queries, test_targets):
    results = cosine_search(query_emb, top_k=max(ks))

    # Precision@k
    for k in ks:
        if target_idx in results[:k]:
            precision[k] += 1

    # MRR
    if target_idx in results:
        rank = results.index(target_idx) + 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    # Косинусное расстояние до правильного ответа
    q = query_emb / np.linalg.norm(query_emb)
    sim = float(embeddings_norm[target_idx] @ q)
    top5_distances.append(1 - sim)   # distance = 1 - similarity

n = len(test_queries)
print('=== Метрики качества поиска ===')
for k in ks:
    print(f'Precision@{k}: {precision[k]/n:.3f}  ({precision[k]}/{n})')
print(f'MRR:          {np.mean(reciprocal_ranks):.3f}')
print(f'Avg distance: {np.mean(top5_distances):.3f}')

## 3. Визуализация метрик и расстояний

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# — Precision@k bar chart —
ax = axes[0]
k_labels = [f'P@{k}' for k in ks] + ['MRR']
values   = [precision[k]/n for k in ks] + [np.mean(reciprocal_ranks)]
bars = ax.bar(k_labels, values, color=['steelblue']*3 + ['tomato'], alpha=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Метрики качества поиска')
ax.set_ylabel('Значение')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

# — Распределение косинусных расстояний —
ax = axes[1]
ax.hist(top5_distances, bins=20, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(np.mean(top5_distances), color='tomato', linestyle='--',
           label=f'Среднее: {np.mean(top5_distances):.3f}')
ax.axvline(0.4, color='orange', linestyle=':', label='Порог relevance (0.4)')
ax.set_xlabel('Косинусное расстояние (запрос → правильный ответ)')
ax.set_ylabel('Количество запросов')
ax.set_title('Распределение расстояний')
ax.legend()

plt.tight_layout()
plt.savefig('rag_metrics.png', dpi=150)
plt.show()

## 4. Анализ ошибок — где поиск промахивается

In [ ]:
print('=== Случаи когда правильный ответ НЕ попал в топ-1 ===\n')
errors = []
for i, (query_emb, target_idx) in enumerate(zip(test_queries, test_targets)):
    results = cosine_search(query_emb, top_k=5)
    if results[0] != target_idx:
        errors.append((i, target_idx, results[0], results))

print(f'Ошибок P@1: {len(errors)} из {n} ({len(errors)/n:.1%})\n')

for err_i, (i, target_idx, top1_idx, results) in enumerate(errors[:5]):
    query_words = texts[target_idx].split()
    query_text  = ' '.join(query_words[:max(5, len(query_words)*2//5)])
    print(f'--- Ошибка {err_i+1} ---')
    print(f'Запрос:        "{query_text[:80]}…"')
    print(f'Ожидалось:     "{texts[target_idx][:80]}…"')
    print(f'Получено (1):  "{texts[top1_idx][:80]}…"')
    rank = results.index(target_idx) + 1 if target_idx in results else '>5'
    print(f'Ранг правильного ответа: {rank}\n')

## 5. Итоги и выводы

In [ ]:
mrr = np.mean(reciprocal_ranks)
p1  = precision[1] / n
p5  = precision[5] / n

print('=== Итоговая оценка качества RAG ===')
print(f'Тест-сет: {n} синтетических запросов')
print(f'Precision@1: {p1:.2%} — правильный ответ на первом месте')
print(f'Precision@5: {p5:.2%} — правильный ответ в топ-5')
print(f'MRR:         {mrr:.3f} — средний обратный ранг')
print()
if p5 >= 0.8:
    print('✅ Поиск работает хорошо: >80% правильных ответов в топ-5')
elif p5 >= 0.6:
    print('⚠️  Поиск работает удовлетворительно: 60-80% в топ-5')
else:
    print('❌ Поиск требует улучшений: <60% в топ-5')
print()
print('Порог релевантности в боте: cosine_distance < 0.4')
below_threshold = sum(1 for d in top5_distances if d < 0.4)
print(f'Запросов с расстоянием < 0.4: {below_threshold}/{n} ({below_threshold/n:.1%})')